# Gemma 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/gemma.ipynb)

The same extraction task on Google's Gemma 4 (Apache-2.0): the E4B (4.5B effective, a 280-token vision encoder) and the 12B (the unified, encoder-free model). First the base models' screens, as the Qwen bases were screened before any training (the bare prompt, the shown examples and the specification forms over the same 60 pictures and 40 specifications, thinking off and on), so the expectation is set before an arm trains; then the arms under Lucia v1's data and training properties verbatim (rank 32, alpha 64, two epochs, lr 2e-4 cosine, batch 8 x 2, plain Adam, seed 20261030) on the language model's projections, thinking off through the template's own switch, 280 image tokens a picture (the models' default budget); each arm's screen, then the two held-out reads with the adapter it trained, unattended. The reads print beside Lucia's so the two families are read on one ruler. The card works through `runs/queue_gemma.json` in the project repo ([AbstractPhil/qwen-symbolic-thinking-image-training](https://huggingface.co/AbstractPhil/qwen-symbolic-thinking-image-training)); each run leaves `runs/done/<name>.json`, and the cell ends after 45 idle minutes. The base model is [google/gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it).

Runtime: a GPU (the RTX PRO 6000 or an A100; the small bases fit smaller cards at batch 8) and the `HF_TOKEN` secret with write access to the project repo. Run C1, restart the session (Runtime > Restart session), then run C2.


In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.21", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (training still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: work through the queue (runs/queue_gemma.json on the Hub; new runs can be added while this runs)
from qwen_symbolic_thinking.queue import run_queue

ran = run_queue("gemma")
print("ran:", ran)
